<a href="https://colab.research.google.com/github/annna-martynova/A-B-test-calculation/blob/main/AB_test_calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#installing and updating the Google Cloud BigQuery client library
!pip install --upgrade google-cloud-bigquery

In [ ]:
#installing all necessary libraries
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
from google.cloud.bigquery.job import query
#authenticating
auth.authenticate_user()
client = bigquery.Client(project="data-analytics-mate")

query = """
WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      p.country,
      p.device,
      p.continent,
      p.channel,
      a.test,
      a.test_group
    FROM `DA.ab_test` a
    JOIN `DA.session` s
      ON a.ga_session_id = s.ga_session_id
    JOIN `DA.session_params` p
      ON p.ga_session_id = a.ga_session_id
  ),
  sessions_with_orders AS (
    SELECT
      s.date,
      s.country,
      s.device,
      s.continent,
      s.channel,
      s.test,
      s.test_group,
      COUNT(DISTINCT o.ga_session_id) AS value
    FROM `DA.order` o
    JOIN session_info s
      ON o.ga_session_id = s.ga_session_id
    GROUP BY ALL
  ),
  events AS (
    SELECT
      s.date,
      s.country,
      s.device,
      s.continent,
      s.channel,
      s.test,
      s.test_group,
      COUNT(DISTINCT e.ga_session_id) AS value,
      e.event_name
    FROM `DA.event_params` e
    JOIN session_info s
      ON e.ga_session_id = s.ga_session_id
    GROUP BY ALL
  ),
  session AS (
    SELECT
      s.date,
      s.country,
      s.device,
      s.continent,
      s.channel,
      s.test,
      s.test_group,
      COUNT(DISTINCT s.ga_session_id) AS value
    FROM session_info s
    GROUP BY ALL
  ),
  account AS (
    SELECT
      s.date,
      s.country,
      s.device,
      s.continent,
      s.channel,
      s.test,
      s.test_group,
      COUNT(DISTINCT a.ga_session_id) AS value
    FROM `DA.account_session` a
    JOIN session_info s
      ON a.ga_session_id = s.ga_session_id
    GROUP BY ALL
  )

SELECT
  *,
  'session with orders' AS event_name
FROM sessions_with_orders
UNION ALL
SELECT *
FROM events
UNION ALL
SELECT
  *,
  'session' AS event_name
FROM session
UNION ALL
SELECT
  *,
  'new_account' AS event_name
FROM account

"""

query_job = client.query(query).result()
ab_testing = query_job.to_dataframe()
ab_testing.tail()

,date,country,device,continent,channel,test,test_group,value,event_name
800991,2020-11-17,Vietnam,desktop,Asia,Social Search,1,1,1,user_engagement
800992,2020-12-08,Vietnam,mobile,Asia,Organic Search,4,1,1,first_visit
800993,2020-12-18,Vietnam,mobile,Asia,Direct,3,2,1,page_view
800994,2020-12-20,Vietnam,desktop,Asia,Direct,3,2,1,first_visit
800995,2021-01-12,Vietnam,desktop,Asia,Paid Search,4,2,1,view_promotion


add_payment_info / session\
add_shipping_info / session\
begin_checkout / session\
new_accounts / session\

In [ ]:
ab_testing['date'] = pd.to_datetime(ab_testing['date'])
ab_testing['value'] = ab_testing['value'].astype(int)
ab_testing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        800996 non-null  datetime64[ns]
 1   country     800996 non-null  object        
 2   device      800996 non-null  object        
 3   continent   800996 non-null  object        
 4   channel     800996 non-null  object        
 5   test        800996 non-null  Int64         
 6   test_group  800996 non-null  Int64         
 7   value       800996 non-null  int64         
 8   event_name  800996 non-null  object        
dtypes: Int64(2), datetime64[ns](1), int64(1), object(5)
memory usage: 56.5+ MB


In [ ]:
metrics = ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_account']

In [ ]:
def agg_data(df, group_columns):
  metrics_cnt = df.groupby(group_columns + ['event_name'])['value'].agg('sum').reset_index()
  den_df = metrics_cnt[metrics_cnt['event_name'] == 'session']
  metrics_cnt = metrics_cnt[metrics_cnt['event_name'].isin(metrics)]
  testing_df = pd.merge(metrics_cnt, den_df, how='left', on=group_columns)
  testing_df = testing_df.rename(columns={
    'event_name_x': 'numerator_name',
    'value_x':      'numerator',
    'event_name_y': 'denominator_name',
    'value_y':      'denominator'
})
  return testing_df

In [ ]:
def calculated_significance(testing_df, segment_col=None):
  control_group = testing_df['test_group'].min()
  control_df = testing_df[testing_df['test_group'] == control_group]
  test_df = testing_df[testing_df['test_group'] != control_group]
  result = []
  for index, row in test_df.iterrows():
    control_row = control_df[(control_df['test'] == row['test']) & (control_df['numerator_name'] == row['numerator_name'])]
    if len(control_row) == 0:
      continue
    conversion_control = control_row.iloc[0]['numerator'] / control_row.iloc[0]['denominator']
    conversion_test = row['numerator'] / row['denominator']
    metric_change = (conversion_test - conversion_control) / conversion_control
    z_stat, p_value = proportions_ztest(
                      count = [row['numerator'], control_row.iloc[0]['numerator']],
                      nobs = [row['denominator'], control_row.iloc[0]['denominator']])
    significant = p_value < 0.05
    result.append({
        'test_number' : row['test'],
        'metric' : row['numerator_name'] + '/' + row['denominator_name'],
        'numerator_event': row['numerator_name'],
        'denominator_event': row['denominator_name'],
        'numerator_control': control_row.iloc[0]['numerator'],
        'denominator_control': control_row.iloc[0]['denominator'],
        'conversion_rate_control': conversion_control,
        'numerator_test': row['numerator'],
        'denominator_test': row['denominator'],
        'conversion_rate_test': conversion_test,
        'metric_change': metric_change,
        'z_stat': z_stat,
        'p_value': p_value,
        'significant': significant,
        'segment_value': row[segment_col] if segment_col else 'total'
    })
  return pd.DataFrame(result)

In [ ]:
aggregated = agg_data(ab_testing,['test', 'test_group'])

total_result_df = calculated_significance(aggregated)
total_result_df

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant,segment_value
0,1,add_payment_info/session,add_payment_info,session,964,45362,0.021251,983,45193,0.021751,0.023523,0.518551,0.604074,False,total
1,1,add_shipping_info/session,add_shipping_info,session,1860,45362,0.041003,1992,45193,0.044078,0.074973,2.291930,0.021910,True,total
2,1,begin_checkout/session,begin_checkout,session,1862,45362,0.041048,1992,45193,0.044078,0.073818,2.258498,0.023915,True,total
3,1,new_account/session,new_account,session,3823,45362,0.084278,3681,45193,0.081451,-0.033543,-1.542883,0.122859,False,total
4,2,add_payment_info/session,add_payment_info,session,1096,50637,0.021644,1125,50244,0.022391,0.034489,0.807895,0.419151,False,total
5,2,add_shipping_info/session,add_shipping_info,session,2194,50637,0.043328,2136,50244,0.042513,-0.018821,-0.638943,0.522860,False,total
6,2,begin_checkout/session,begin_checkout,session,2195,50637,0.043348,2137,50244,0.042532,-0.018809,-0.638682,0.523030,False,total
7,2,new_account/session,new_account,session,4165,50637,0.082252,4184,50244,0.083274,0.012419,0.588793,0.556000,False,total
8,3,add_payment_info/session,add_payment_info,session,1771,70047,0.025283,1788,70439,0.025384,0.003981,0.120029,0.904461,False,total
9,3,add_shipping_info/session,add_shipping_info,session,2915,70047,0.041615,2889,70439,0.041014,-0.014435,-0.565667,0.571620,False,total


In [ ]:
segments = {
    'total':   ['test', 'test_group'],
    'device':  ['test', 'test_group', 'device'],
    'channel' : ['test', 'test_group', 'channel'],
    'country': ['test', 'test_group', 'country'],
    'continent': ['test', 'test_group', 'continent'],
    'device_by_channel':  ['test', 'test_group', 'device', 'channel']
  }

def segment_test(df, segments):
    segment_result = []
    for segment_name, group_columns in segments.items():
        base_columns = ['test', 'test_group']
        segment_columns = [c for c in group_columns if c not in base_columns]
        segment_col = segment_columns[0] if segment_columns else None

        test_for_segment = agg_data(df, group_columns)
        calculated_segment = calculated_significance(test_for_segment, segment_col=segment_col)
        calculated_segment['segment'] = segment_name
        segment_result.append(calculated_segment)
    return pd.concat(segment_result)

In [ ]:
segment_result_df = segment_test(ab_testing, segments)
segment_result_df

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant,segment_value,segment
0,1,add_payment_info/session,add_payment_info,session,964,45362,0.021251,983,45193,0.021751,0.023523,0.518551,0.604074,False,total,total
1,1,add_shipping_info/session,add_shipping_info,session,1860,45362,0.041003,1992,45193,0.044078,0.074973,2.291930,0.021910,True,total,total
2,1,begin_checkout/session,begin_checkout,session,1862,45362,0.041048,1992,45193,0.044078,0.073818,2.258498,0.023915,True,total,total
3,1,new_account/session,new_account,session,3823,45362,0.084278,3681,45193,0.081451,-0.033543,-1.542883,0.122859,False,total,total
4,2,add_payment_info/session,add_payment_info,session,1096,50637,0.021644,1125,50244,0.022391,0.034489,0.807895,0.419151,False,total,total
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,4,new_account/session,new_account,session,1269,14371,0.088303,17,188,0.090426,0.024039,0.101899,0.918837,False,tablet,device_by_channel
236,4,add_payment_info/session,add_payment_info,session,239,14371,0.016631,3,120,0.025000,0.503243,0.712479,0.476168,False,tablet,device_by_channel
237,4,add_shipping_info/session,add_shipping_info,session,363,14371,0.025259,3,120,0.025000,-0.010262,-0.018021,0.985622,False,tablet,device_by_channel
238,4,begin_checkout/session,begin_checkout,session,363,14371,0.025259,3,120,0.025000,-0.010262,-0.018021,0.985622,False,tablet,device_by_channel


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder_path = "/content/drive/My Drive/DA_portfolio/"
file_name = 'ab_testing_calcs_total.csv'

total_result_df.to_csv(folder_path + file_name, index=False)

print(f"Saved to: {folder_path + file_name}")

Mounted at /content/drive
Saved to: /content/drive/My Drive/DA_portfolio/ab_testing_calcs_total.csv


In [ ]:
folder_path = "/content/drive/My Drive/DA_portfolio/"
file_name = 'ab_testing_calcs_segments.csv'

segment_result_df.to_csv(folder_path + file_name, index=False)

print(f"Saved to: {folder_path + file_name}")

Saved to: /content/drive/My Drive/DA_portfolio/ab_testing_calcs_segments.csv


###**Tableau visualization:** https://public.tableau.com/app/profile/anna.martynova/viz/ABTestingSignificanceAnalysis/ABTestingSignificanceAnalysis#2

##**Conclution**

**Project Goal:**\
Automation of A/B testing calculations for various metrics and segments using Python, with result visualization in Tableau.

**Statistical Aproach:**\
Conducting a z-test of proportions for the control and test groups at a statistical significance level of α = 0.05.

**Main Metrics:**
- add_payment_info / session
- add_shipping_info / session
- begin_checkout / session
- new_accounts / session

**Additional levels of detail:**
- device
- channel
- country
- continent

##**Results**
Test 1 showed the most positive changes, 3 out of 4 key metrics improved significantly.

Tests 2 & 3 did not show statistically significant results. \
Based on the analysis, these hypotheses were not confirmed, and the proposed changes are not recommended for implementation.

Test 4  showed mixed results, with 2 key metrics showing statistically significant changes.\
Further analysis is required to determine whether effects are strong enough to compensate for weaker performance in other metrics.

##**Recomendations**

1. Based on the results of Test 1, the changes should be implemented across all segments.
2. Test 4 requires further refinement and additional analysis.
3. Tests 2 and 3 should be rejected, as the key metrics did not show any significant changes.


